In [7]:
import gymnasium as gym
import flax
import jax
import jax.numpy as jnp
import ml_collections
import optax
import ogbench
import tyro
import numpy as np

In [2]:
dataset_id = 'pointmaze-medium-navigate-v0'             # 'pointmaze-large-navigate-v0'

In [3]:
env, train_dataset, val_dataset = ogbench.make_env_and_datasets(dataset_id)

In [4]:
def get_config():
    config = ml_collections.ConfigDict(
        dict(
            # Agent hyperparameters.
            agent_name = 'cfgbc',  # Agent name.
            lr = 3e-4,  # Learning rate.
            batch_size = 32,  # Batch size.
            actor_hidden_dims = (64, 128, 64),  # Actor network hidden dimensions.
            discount = 0.99,  # Discount factor (unused by default; can be used for geometric goal sampling in GCDataset).
            const_std = True,  # Whether to use constant standard deviation for the actor.
            discrete = False,  # Whether the action space is discrete.
            encoder = None,  # Visual encoder name (None, 'impala_small', etc.).
            # encoder=ml_collections.config_dict.placeholder(str),

            # Dataset hyperparameters.
            dataset_class = 'GCDataset',  # Dataset class name.
            value_p_curgoal = 0.0,  # Unused (defined for compatibility with GCDataset).
            value_p_trajgoal = 1.0,  # Unused (defined for compatibility with GCDataset).
            value_p_randomgoal = 0.0,  # Unused (defined for compatibility with GCDataset).
            value_geom_sample = False,  # Unused (defined for compatibility with GCDataset).
            actor_p_curgoal = 0.0,  # Probability of using the current state as the actor goal.
            actor_p_trajgoal = 1.0,  # Probability of using a future state in the same trajectory as the actor goal.
            actor_p_randomgoal = 0.0,  # Probability of using a random state as the actor goal.
            actor_geom_sample = False,  # Whether to use geometric sampling for future actor goals.
            gc_negative = False,  # Unused (defined for compatibility with GCDataset).
            p_aug = 0.0,  # Probability of applying image augmentation.
            frame_stack = None,  # Number of frames to stack.
            # frame_stack = ml_collections.config_dict.placeholder(int),
        )
    )
    return config

In [ ]:
class CFGBCAgent(flax.struct.PyTreeNode):
    pass

In [56]:
print(env.observation_space)
print(env.action_space)

Box(-inf, inf, (2,), float64)
Box(-1.0, 1.0, (2,), float32)


In [54]:
print(env.spec.max_episode_steps)

1000


In [42]:
obs, info = env.reset()
print(obs, info)

[ 8.64013854 16.06295326] {'goal': array([ 3.25959597, 11.63521974])}


In [47]:
print(type(obs[0]))

<class 'numpy.float64'>


In [5]:
print(train_dataset.keys())

dict_keys(['observations', 'actions', 'terminals', 'next_observations'])


In [6]:
print(len(train_dataset['observations']))
print(len(train_dataset['actions']))
print(len(train_dataset['terminals']))
print(len(train_dataset['next_observations']))

1000000
1000000
1000000
1000000


In [58]:
print(train_dataset['observations'][998:1002])

[[11.664439   9.753045 ]
 [11.69234    9.553045 ]
 [ 3.1425061  8.577186 ]
 [ 3.0999362  8.777185 ]]


In [59]:
print(train_dataset['terminals'][998:1002])

[0. 1. 0. 0.]


In [60]:
print(train_dataset['next_observations'][998:1002])

[[11.69234    9.553045 ]
 [11.591223   9.353045 ]
 [ 3.0999362  8.777185 ]
 [ 3.1995738  8.969827 ]]


In [61]:
print(train_dataset['actions'][998:1002])

[[ 0.13949911 -1.        ]
 [-0.5055844  -1.        ]
 [-0.21284929  1.        ]
 [ 0.4981876   0.9632062 ]]


In [35]:
print(train_dataset['next_observations'][:100])

[[12.443868  12.768156 ]
 [12.531055  12.968156 ]
 [12.395658  13.06429  ]
 [12.514426  13.26429  ]
 [12.489343  13.46429  ]
 [12.443696  13.664289 ]
 [12.510444  13.739171 ]
 [12.363752  13.848236 ]
 [12.321012  14.024125 ]
 [12.189367  14.088035 ]
 [11.989367  14.166822 ]
 [11.89445   14.294525 ]
 [11.69445   14.275756 ]
 [11.564189  14.373608 ]
 [11.459133  14.4338   ]
 [11.367556  14.43445  ]
 [11.167556  14.441598 ]
 [10.967556  14.452037 ]
 [10.855611  14.58689  ]
 [10.655611  14.525921 ]
 [10.455611  14.725921 ]
 [10.255611  14.819965 ]
 [10.055611  15.019965 ]
 [ 9.908755  15.219965 ]
 [ 9.941653  15.237059 ]
 [ 9.859776  15.437058 ]
 [ 9.659776  15.4630375]
 [ 9.66244   15.471275 ]
 [ 9.564924  15.671275 ]
 [ 9.65291   15.871275 ]
 [ 9.688248  16.048426 ]
 [ 9.624505  16.248425 ]
 [ 9.574268  16.338575 ]
 [ 9.522549  16.507381 ]
 [ 9.428487  16.707382 ]
 [ 9.445872  16.907381 ]
 [ 9.270991  16.956793 ]
 [ 9.21297   17.156794 ]
 [ 9.222528  17.307276 ]
 [ 9.033621  17.507277 ]


In [25]:
count = 0

for i, j in enumerate(train_dataset['terminals']):
    print(j)
    count += 1
    if count > 2000:
        break

# print(count)

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [28]:
count = 0

for i, j in enumerate(val_dataset['terminals']):
    if j == 1.0:
        count += 1

print(count)

100


In [32]:
print(train_dataset['observations'])

[[12.429421  12.568156 ]
 [12.443868  12.768156 ]
 [12.531055  12.968156 ]
 ...
 [ 4.1317916  4.634845 ]
 [ 3.9317915  4.8348446]
 [ 4.0947647  4.899703 ]]


In [33]:
print(train_dataset['next_observations'])

[[12.443868  12.768156 ]
 [12.531055  12.968156 ]
 [12.395658  13.06429  ]
 ...
 [ 3.9317915  4.8348446]
 [ 4.0947647  4.899703 ]
 [ 4.1459765  4.962475 ]]


In [31]:
env.reset()
print(env)

<TimeLimit<OrderEnforcing<PassiveEnvChecker<MazeEnv<pointmaze-medium-v0>>>>>


In [50]:
sub_steps = 25

In [97]:
print(type(train_dataset))

<class 'dict'>


In [231]:
observations = train_dataset['observations']
actions = train_dataset['actions']
terminals = train_dataset['terminals']

(terminal_locs,) = np.nonzero(terminals > 0)
starts = np.concatenate([[0], terminal_locs[:-1] + 1])
starts = np.append(starts, len(observations))
length = int(np.diff(np.concatenate([[-1], terminal_locs]))[0])

new_starts = []
segment_length = sub_steps + 1

for i in range(len(starts) - 1):
    start = starts[i]
    end = starts[i + 1]

    segment = np.arange(start, end, sub_steps, dtype=np.int32)
    new_starts.append(segment)

starts = np.concatenate(new_starts)

In [206]:
print(terminal_locs)

[   999   1999   2999   3999   4999   5999   6999   7999   8999   9999
  10999  11999  12999  13999  14999  15999  16999  17999  18999  19999
  20999  21999  22999  23999  24999  25999  26999  27999  28999  29999
  30999  31999  32999  33999  34999  35999  36999  37999  38999  39999
  40999  41999  42999  43999  44999  45999  46999  47999  48999  49999
  50999  51999  52999  53999  54999  55999  56999  57999  58999  59999
  60999  61999  62999  63999  64999  65999  66999  67999  68999  69999
  70999  71999  72999  73999  74999  75999  76999  77999  78999  79999
  80999  81999  82999  83999  84999  85999  86999  87999  88999  89999
  90999  91999  92999  93999  94999  95999  96999  97999  98999  99999
 100999 101999 102999 103999 104999 105999 106999 107999 108999 109999
 110999 111999 112999 113999 114999 115999 116999 117999 118999 119999
 120999 121999 122999 123999 124999 125999 126999 127999 128999 129999
 130999 131999 132999 133999 134999 135999 136999 137999 138999 139999
 14099

In [232]:
print(len(starts))

40000


In [233]:
print(starts[:200])

[   0   25   50   75  100  125  150  175  200  225  250  275  300  325
  350  375  400  425  450  475  500  525  550  575  600  625  650  675
  700  725  750  775  800  825  850  875  900  925  950  975 1000 1025
 1050 1075 1100 1125 1150 1175 1200 1225 1250 1275 1300 1325 1350 1375
 1400 1425 1450 1475 1500 1525 1550 1575 1600 1625 1650 1675 1700 1725
 1750 1775 1800 1825 1850 1875 1900 1925 1950 1975 2000 2025 2050 2075
 2100 2125 2150 2175 2200 2225 2250 2275 2300 2325 2350 2375 2400 2425
 2450 2475 2500 2525 2550 2575 2600 2625 2650 2675 2700 2725 2750 2775
 2800 2825 2850 2875 2900 2925 2950 2975 3000 3025 3050 3075 3100 3125
 3150 3175 3200 3225 3250 3275 3300 3325 3350 3375 3400 3425 3450 3475
 3500 3525 3550 3575 3600 3625 3650 3675 3700 3725 3750 3775 3800 3825
 3850 3875 3900 3925 3950 3975 4000 4025 4050 4075 4100 4125 4150 4175
 4200 4225 4250 4275 4300 4325 4350 4375 4400 4425 4450 4475 4500 4525
 4550 4575 4600 4625 4650 4675 4700 4725 4750 4775 4800 4825 4850 4875
 4900 

In [91]:
print(observations[999975:999975+sub_steps])
print(len(observations[999975:999975+sub_steps]))

[[3.9691904 1.547768 ]
 [3.9827821 1.347768 ]
 [4.015392  1.4342028]
 [3.8411968 1.6342027]
 [4.041197  1.8342028]
 [4.0640182 1.9615217]
 [4.0427165 1.9466218]
 [4.081431  2.1466217]
 [3.9640703 2.3466218]
 [4.1215053 2.4806654]
 [4.238449  2.6806655]
 [4.217737  2.8023882]
 [4.0724297 3.0023882]
 [4.12577   3.2023883]
 [4.1034307 3.313696 ]
 [4.162275  3.4943035]
 [3.9912481 3.6943033]
 [3.9407146 3.8943033]
 [3.9408271 4.0943036]
 [3.7788975 4.2943034]
 [3.7319245 4.3381124]
 [3.9317915 4.5381126]
 [4.1317916 4.634845 ]
 [3.9317915 4.8348446]
 [4.0947647 4.899703 ]]
25


In [175]:
print(actions[999975:999975+sub_steps])
print(len(actions[999975:999975+sub_steps]))

[[ 6.79588616e-02 -1.00000000e+00]
 [ 1.63048655e-01  4.32173997e-01]
 [-8.70976210e-01  1.00000000e+00]
 [ 1.00000000e+00  1.00000000e+00]
 [ 1.14107512e-01  6.36594892e-01]
 [-1.06508240e-01 -7.44996294e-02]
 [ 1.93571076e-01  1.00000000e+00]
 [-5.86801827e-01  1.00000000e+00]
 [ 7.87174344e-01  6.70218110e-01]
 [ 5.84719360e-01  1.00000000e+00]
 [-1.03559218e-01  6.08614266e-01]
 [-7.26539135e-01  1.00000000e+00]
 [ 2.66703039e-01  1.00000000e+00]
 [-1.11697294e-01  5.56537747e-01]
 [ 2.94220209e-01  9.03037846e-01]
 [-8.55132818e-01  1.00000000e+00]
 [-2.52666742e-01  1.00000000e+00]
 [ 5.61953115e-04  1.00000000e+00]
 [-8.09647679e-01  1.00000000e+00]
 [-2.34865323e-01  2.19045848e-01]
 [ 9.99334693e-01  1.00000000e+00]
 [ 1.00000000e+00  4.83661324e-01]
 [-1.00000000e+00  1.00000000e+00]
 [ 8.14866126e-01  3.24291587e-01]
 [ 2.56058693e-01  3.13858688e-01]]
25


In [96]:
test = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
print(test[:-1])

[0, 1, 2, 3, 4, 5, 6, 7, 8]


In [221]:
import dataclasses


@dataclasses.dataclass
class TrajectoryGCDataset:

    dataset: dict
    sub_steps: int

    def __post_init__(self):

        self.observations = self.dataset['observations']
        self.actions = self.dataset['actions']
        self.terminals = self.dataset['terminals']

        (terminal_locs,) = np.nonzero(self.terminals > 0)
        self.starts = np.concatenate([[0], terminal_locs[:-1] + 1])
        self.starts = np.append(self.starts, len(self.observations))

        self.segment_length = sub_steps + 1
        new_starts = []

        for i in range(len(self.starts) - 1):
            start = self.starts[i]
            end = self.starts[i + 1]

            segment = np.arange(start, end, sub_steps, dtype=np.int32)
            new_starts.append(segment)

        self.starts = np.concatenate(new_starts)

    def get_random_idxs(self, num_idxs):
        num_trajectory = self.starts.shape[0]
        return np.random.randint(num_trajectory, size=num_idxs)

    def sample(self, batch_size, idxs=None, evaluation=False):
        if idxs is None:
            idxs = self.get_random_idxs(batch_size)

        idxs = np.array(idxs, dtype=np.int32)
        starts = self.starts[idxs]

        idx_states = starts[:, None] + np.arange(self.segment_length, dtype=np.int32)[None, :]  # (B, L)

        return dict(
            observations = jnp.array(self.observations[idx_states]),
            next_observations = jnp.array(self.observations[idx_states]),
            actions = jnp.array(self.actions[idx_states]),
            rewards = jnp.array(self.terminals[idx_states]).at[:, -1].set(1)
        )

In [222]:
trajectory_dataset = TrajectoryGCDataset(train_dataset, sub_steps=25)

In [223]:
sample = trajectory_dataset.sample(1)
B, n = sample['observations'].shape[0], sample['observations'].shape[1] - 1

In [224]:
print(B, n)

1 25


In [225]:
print(sample['observations'][0])
print(len(sample['observations'][0]))

[[11.487995  10.033451 ]
 [11.419493   9.908773 ]
 [11.43747    9.708774 ]
 [11.457824   9.508774 ]
 [11.559175   9.308773 ]
 [11.596839   9.108773 ]
 [11.510537   9.018538 ]
 [11.449686   8.818539 ]
 [11.635493   8.716337 ]
 [11.741385   8.598588 ]
 [11.757072   8.398588 ]
 [11.705909   8.198588 ]
 [11.678013   7.998588 ]
 [11.7933855  7.980722 ]
 [11.887628   7.780722 ]
 [11.946101   7.580722 ]
 [11.981848   7.4431396]
 [12.098198   7.2431393]
 [12.154527   7.0617666]
 [12.088769   6.8617663]
 [11.888769   6.842566 ]
 [11.905821   6.642566 ]
 [11.89395    6.451116 ]
 [11.98342    6.2511163]
 [12.051893   6.0511165]
 [12.007976   5.9152665]]
26


In [226]:
print(sample['actions'][0])
print(len(sample['actions'][0]))

[[-0.34251285 -0.62339056]
 [ 0.08988584 -1.        ]
 [ 0.10176984 -1.        ]
 [ 0.5067517  -1.        ]
 [ 0.1883214  -1.        ]
 [-0.43150973 -0.45117363]
 [-0.30425608 -1.        ]
 [ 0.92903984 -0.51100826]
 [ 0.5294589  -0.5887448 ]
 [ 0.07843329 -1.        ]
 [-0.25581595 -1.        ]
 [-0.13948125 -1.        ]
 [ 0.5768667  -0.08933013]
 [ 0.4712095  -1.        ]
 [ 0.29236713 -1.        ]
 [ 0.17873417 -0.687913  ]
 [ 0.58175    -1.        ]
 [ 0.2816444  -0.9068646 ]
 [-0.3287889  -1.        ]
 [-1.         -0.09600188]
 [ 0.08525945 -1.        ]
 [-0.05935368 -0.95724934]
 [ 0.4473517  -1.        ]
 [ 0.3423638  -1.        ]
 [-0.21958978 -0.679249  ]
 [ 1.          0.8346425 ]]
26


In [227]:
print(sample['observations'][0, :n])
print(len(sample['observations'][0, :n]))

[[11.487995  10.033451 ]
 [11.419493   9.908773 ]
 [11.43747    9.708774 ]
 [11.457824   9.508774 ]
 [11.559175   9.308773 ]
 [11.596839   9.108773 ]
 [11.510537   9.018538 ]
 [11.449686   8.818539 ]
 [11.635493   8.716337 ]
 [11.741385   8.598588 ]
 [11.757072   8.398588 ]
 [11.705909   8.198588 ]
 [11.678013   7.998588 ]
 [11.7933855  7.980722 ]
 [11.887628   7.780722 ]
 [11.946101   7.580722 ]
 [11.981848   7.4431396]
 [12.098198   7.2431393]
 [12.154527   7.0617666]
 [12.088769   6.8617663]
 [11.888769   6.842566 ]
 [11.905821   6.642566 ]
 [11.89395    6.451116 ]
 [11.98342    6.2511163]
 [12.051893   6.0511165]]
25


In [228]:
print(sample['next_observations'][0, 1:])
print(len(sample['next_observations'][0, 1:]))

[[11.419493   9.908773 ]
 [11.43747    9.708774 ]
 [11.457824   9.508774 ]
 [11.559175   9.308773 ]
 [11.596839   9.108773 ]
 [11.510537   9.018538 ]
 [11.449686   8.818539 ]
 [11.635493   8.716337 ]
 [11.741385   8.598588 ]
 [11.757072   8.398588 ]
 [11.705909   8.198588 ]
 [11.678013   7.998588 ]
 [11.7933855  7.980722 ]
 [11.887628   7.780722 ]
 [11.946101   7.580722 ]
 [11.981848   7.4431396]
 [12.098198   7.2431393]
 [12.154527   7.0617666]
 [12.088769   6.8617663]
 [11.888769   6.842566 ]
 [11.905821   6.642566 ]
 [11.89395    6.451116 ]
 [11.98342    6.2511163]
 [12.051893   6.0511165]
 [12.007976   5.9152665]]
25


In [229]:
print(sample['actions'][0, :n])
print(len(sample['actions'][0, :n]))

[[-0.34251285 -0.62339056]
 [ 0.08988584 -1.        ]
 [ 0.10176984 -1.        ]
 [ 0.5067517  -1.        ]
 [ 0.1883214  -1.        ]
 [-0.43150973 -0.45117363]
 [-0.30425608 -1.        ]
 [ 0.92903984 -0.51100826]
 [ 0.5294589  -0.5887448 ]
 [ 0.07843329 -1.        ]
 [-0.25581595 -1.        ]
 [-0.13948125 -1.        ]
 [ 0.5768667  -0.08933013]
 [ 0.4712095  -1.        ]
 [ 0.29236713 -1.        ]
 [ 0.17873417 -0.687913  ]
 [ 0.58175    -1.        ]
 [ 0.2816444  -0.9068646 ]
 [-0.3287889  -1.        ]
 [-1.         -0.09600188]
 [ 0.08525945 -1.        ]
 [-0.05935368 -0.95724934]
 [ 0.4473517  -1.        ]
 [ 0.3423638  -1.        ]
 [-0.21958978 -0.679249  ]]
25


In [230]:
print(sample['rewards'][0, 1:])
print(len(sample['rewards'][0, 1:]))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1.]
25
